In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS real_estate.silver;

In [0]:
df = spark.table('real_estate.bronze.washington_ultimate')

display(df.limit(5))

In [0]:
#cleaning column name
import re 

def clean_column(df):
    new_cols = []
    for c in df.columns:
        c = c.strip()
        c = c.lower()
        c = re.sub(r"[^\w]+", "_", c)
        c = re.sub(r"_+", "_", c)
        c = c.strip('_')
        new_cols.append(c)
    return df.toDF(*new_cols)

df_cleaned_column_name = clean_column(df)

In [0]:
df_cleaned_column_name.printSchema()

In [0]:
#checking null value
from pyspark.sql.functions import col,sum

null_count = df_cleaned_column_name \
    .select([sum(col(c).isNull() \
    .cast('int').alias(c)) for c in df_cleaned_column_name.columns])

display(null_count)

In [0]:
display(df_cleaned_column_name.select('*').where(col('listprice').isNull()&col('lastsoldprice').isNull()))

In [0]:
display(df_cleaned_column_name.select('*').where(col('lastsoldprice').isNull()&col('list_to_sold_ratio').isNull()))

In [0]:
#dropping rows containing both list and last sold price as null
df_drop_na = df_cleaned_column_name.dropna(subset=['listprice', 'lastsoldprice'], how='all')

In [0]:
print(df_cleaned_column_name.count())
print(df_drop_na.count())

In [0]:
display(df_drop_na.select('*').where(col('listprice').isNull()&col('lastsoldprice').isNull()))

In [0]:
display(df_drop_na.select('*').where(col('listprice').isNull()&(col('lastsoldprice').isNull()|col('list_to_sold_ratio').isNull())))

In [0]:
#calculting list_to_sold_ratio if the value is null
from pyspark.sql.functions import when

df_clean = df_drop_na.withColumn(
    'list_to_sold_ratio',
    when(col('list_to_sold_ratio').isNull(), col('lastsoldprice')/col('listprice'))
    .otherwise(col('list_to_sold_ratio'))
    )

In [0]:
null_count = df_clean \
    .select([sum(col(c).isNull() \
    .cast('int').alias(c)) for c in df_clean.columns])

display(null_count)

In [0]:
#calculting listprice if the value is null

df_clean = df_clean.withColumn(
    'listprice',
    when(col('listprice').isNull(), col('lastsoldprice')/col('list_to_sold_ratio'))
    .otherwise(col('listprice'))
    )

In [0]:
null_count = df_clean \
    .select([sum(col(c).isNull() \
    .cast('int').alias(c)) for c in df_clean.columns])

display(null_count)

In [0]:
#calculting last sold price if the value is null

df_clean = df_clean.withColumn(
    'lastsoldprice',
    when(col('lastsoldprice').isNull(), col('listprice')*col('list_to_sold_ratio'))
    .otherwise(col('lastsoldprice'))
    )

df_clean = df_clean.withColumn(
    'lastsoldprice',
    when(col('lastsoldprice').isNull(), col('price_per_sqft')*col('sqft'))
    .otherwise(col('lastsoldprice'))
    )

In [0]:
null_count = df_clean \
    .select([sum(col(c).isNull() \
    .cast('int').alias(c)) for c in df_clean.columns])

display(null_count)

In [0]:
#calculating price per sqft if the value is null

df_clean = df_clean.withColumn(
    'price_per_sqft',
    when(col('price_per_sqft').isNull(), col('lastsoldprice')/col('sqft'))
    .otherwise(col('price_per_sqft'))
    )

In [0]:
null_count = df_clean \
    .select([sum(col(c).isNull() \
    .cast('int').alias(c)) for c in df_clean.columns])

display(null_count)

In [0]:
#calculating sqft if value is null

df_clean = df_clean.withColumn(
    'sqft',
    when(col('sqft').isNull(), col('lastsoldprice')/col('price_per_sqft'))
    .otherwise(col('sqft'))
    )

In [0]:
null_count = df_clean \
    .select([sum(col(c).isNull() \
    .cast('int').alias(c)) for c in df_clean.columns])

display(null_count)

In [0]:
df_clean.printSchema()

In [0]:
df_clean = df_clean.fillna({
    'stories': 0,
    'beds': 0,
    'baths': 0,
    'baths_full': 0,
    'baths_full_calc': 0,
    'garage': 0
})

In [0]:
null_count = df_clean \
    .select([sum(col(c).isNull() \
    .cast('int').alias(c)) for c in df_clean.columns])

display(null_count)